<a href="https://colab.research.google.com/github/Ramy99999999/Flyrank-machine-learning-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ramy99999999/Flyrank-machine-learning-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will use Logistic Regression as the first learned model. My question is a ranking question: which content items should be reviewed first for low-visibility risk? Logistic Regression gives a probability score that can rank items, while remaining simple enough to interpret. I will compare it with the Week-4 inverse-volume baseline using Precision@10, Precision@20, and Precision@50 on the same evaluation split.

In [83]:
%pip install -q duckdb huggingface_hub pandas scikit-learn

In [84]:
import os
import duckdb
import pandas as pd

# Get the Hugging Face token from Colab Secrets
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect DuckDB
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Give DuckDB the Hugging Face token without putting it directly in a query
con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])
con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN getvariable('hf_token')
    )
""")

# Warehouse locations
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
DIM_CONTENT = f"{REL}/dim_content.parquet"
DIM_CLIENTS = f"{REL}/dim_clients.parquet"

# Time windows
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Connected to FlyRank warehouse.")
print("Feature window: February 2026")
print("Outcome/label window: March 2026")

Connected to FlyRank warehouse.
Feature window: February 2026
Outcome/label window: March 2026


In [85]:
con.execute(f"""
    SELECT *
    FROM {FEB}
    LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-02-01,client_e547b89c05043229,content_7995404695ee1ffd,True,True,True,False,57,0,1778,...,0,0,0,0,0,0,0,0,0,2026-02
1,2026-02-01,client_e547b89c05043229,content_1eea820697c3b95a,True,True,True,False,13,0,85,...,0,0,0,0,0,0,0,0,0,2026-02
2,2026-02-01,client_e547b89c05043229,content_ccbb253f142217c3,True,True,True,True,59,0,1001,...,0,0,0,0,0,0,0,0,0,2026-02
3,2026-02-01,client_e547b89c05043229,content_ae16a6b9cf64c80a,True,True,True,False,17,0,287,...,0,0,0,0,0,0,0,0,0,2026-02
4,2026-02-01,client_e547b89c05043229,content_acf700633f016e5a,True,True,True,False,6,0,27,...,0,0,0,0,0,0,0,0,0,2026-02


In [86]:
universe_feb = con.execute(f"""
SELECT
    f.client_hash_id,
    f.content_hash_id
FROM {FEB} AS f
JOIN read_parquet('{DIM_CONTENT}') AS d
    ON f.client_hash_id = d.client_hash_id
    AND f.content_hash_id = d.content_hash_id
GROUP BY
    f.client_hash_id,
    f.content_hash_id,
    d.is_published,
    d.content_created_date
HAVING
    SUM(f.gsc_impressions) >= 100
    AND SUM(f.gsc_clicks) >= 3
    AND d.is_published IS TRUE
    AND d.content_created_date <= DATE '2026-02-28'
""").df()


In [87]:
print("Shape:", universe_feb.shape)
display(universe_feb.head())

Shape: (29700, 2)


,client_hash_id,content_hash_id
0,client_e547b89c05043229,content_ccbb253f142217c3
1,client_e547b89c05043229,content_babd931911c9ee33
2,client_e547b89c05043229,content_7275e583711cf60b
3,client_e547b89c05043229,content_c5baa3a03cfccc41
4,client_e547b89c05043229,content_e24453f672279e47


In [88]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_score

RANDOM_STATE = 42

print("Random seed:", RANDOM_STATE)
print("Model: Logistic Regression")
print("Evaluation: Precision@10, Precision@20, Precision@50")

Random seed: 42
Model: Logistic Regression
Evaluation: Precision@10, Precision@20, Precision@50


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a grouped split by client so content from the same client does not appear in both training and evaluation. This is more honest than a random row split because pages from one client can share patterns. The client identifier is used only for splitting and never as a model feature. I will use 80% of clients for training and 20% for evaluation, then compare the model and Week-4 baseline on exactly the same evaluation rows.


In [89]:
universe_feb = con.execute(f"""
SELECT
    f.client_hash_id,
    f.content_hash_id
FROM {FEB} AS f
JOIN read_parquet('{DIM_CONTENT}') AS d
    ON f.client_hash_id = d.client_hash_id
    AND f.content_hash_id = d.content_hash_id
GROUP BY
    f.client_hash_id,
    f.content_hash_id,
    d.is_published,
    d.content_created_date
HAVING
    SUM(f.gsc_impressions) >= 100
    AND SUM(f.gsc_clicks) >= 3
    AND d.is_published IS TRUE
    AND d.content_created_date <= DATE '2026-02-28'
""").df()

print("universe_feb:", universe_feb.shape)

universe_feb: (29700, 2)


In [90]:
features_feb = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions_feb,
    SUM(gsc_clicks) AS gsc_clicks_feb,
    SUM(
        CASE
            WHEN gsc_data_available IS TRUE
            THEN gsc_sum_position
            ELSE 0
        END
    ) / NULLIF(
        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN gsc_impressions
                ELSE 0
            END
        ),
        0
    ) AS avg_position_feb,
    SUM(sessions_organic) AS sessions_organic_feb,
    SUM(scroll_events) AS scroll_events_feb
FROM {FEB}
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

print("features_feb:", features_feb.shape)

features_feb: (321546, 7)


In [91]:
labels_mar = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS measured_days_mar,
    SUM(gsc_clicks) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS clicks_mar,
    CASE
        WHEN SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) = 0
        THEN 1
        ELSE 0
    END AS went_dark
FROM {MAR}
GROUP BY
    client_hash_id,
    content_hash_id
HAVING
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) > 0
""").df()

print("labels_mar:", labels_mar.shape)

labels_mar: (176738, 5)


In [92]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Load the same warehouse data used in ML-04 / ML-07

# Build the final modeling frame from the same February/March contract logic.

final_frame = (
    features_feb
    .merge(
        universe_feb,
        on=["client_hash_id", "content_hash_id"],
        how="inner"
    )
    .merge(
        labels_mar,
        on=["client_hash_id", "content_hash_id"],
        how="inner"
    )
)

print("Final frame shape:", final_frame.shape)
print("Columns:", final_frame.columns.tolist())

Final frame shape: (29353, 10)
Columns: ['client_hash_id', 'content_hash_id', 'gsc_impressions_feb', 'gsc_clicks_feb', 'avg_position_feb', 'sessions_organic_feb', 'scroll_events_feb', 'measured_days_mar', 'clicks_mar', 'went_dark']


In [93]:
from sklearn.model_selection import GroupShuffleSplit

features = [
    "gsc_impressions_feb",
    "gsc_clicks_feb",
    "avg_position_feb",
    "sessions_organic_feb",
    "scroll_events_feb",
]

X = final_frame[features].copy()
y = final_frame["went_dark"].astype(int)
groups = final_frame["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

test_rows = final_frame.iloc[test_idx].copy()

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train clients:", groups.iloc[train_idx].nunique())
print("Test clients:", groups.iloc[test_idx].nunique())

Train rows: 16391
Test rows: 12962
Train clients: 24
Test clients: 6


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I will train Logistic Regression using only the February features defined by the data contract. The model will produce a probability score for went_dark, which I will use to rank the evaluation rows. The Week-4 inverse-volume baseline will be calculated on the exact same evaluation rows. Both methods will be compared using Precision@10, Precision@20, and Precision@50.


In [94]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

model_scores = model.predict_proba(X_test)[:, 1]

print("Model trained successfully.")
print("Test predictions:", len(model_scores))

Model trained successfully.
Test predictions: 12962


In [95]:
def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return np.asarray(y_true)[order].mean()

# Week-4 baseline:
# inverse log-volume = 1 / log1p(February impressions)
baseline_scores = (
    1 / np.log1p(test_rows["gsc_impressions_feb"].to_numpy())
)

ks = [10, 20, 50]

results = []

for k in ks:
    results.append({
        "K": k,
        "Baseline_Precision@K": precision_at_k(
            y_test.to_numpy(),
            baseline_scores,
            k
        ),
        "LogisticRegression_Precision@K": precision_at_k(
            y_test.to_numpy(),
            model_scores,
            k
        )
    })

comparison = pd.DataFrame(results)

print(comparison.to_string(index=False))

 K  Baseline_Precision@K  LogisticRegression_Precision@K
10                  0.20                            0.10
20                  0.30                            0.15
50                  0.26                            0.22


In [96]:
import numpy as np
import pandas as pd

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return np.asarray(y_true)[order].mean()

# Week-4 inverse-volume baseline on the exact same test rows
baseline_scores = (
    1 / np.log1p(test_rows["gsc_impressions_feb"].to_numpy())
)

ks = [10, 20, 50]

results = []

for k in ks:
    results.append({
        "K": k,
        "Baseline_Precision@K": precision_at_k(
            y_test.to_numpy(),
            baseline_scores,
            k
        ),
        "LogisticRegression_Precision@K": precision_at_k(
            y_test.to_numpy(),
            model_scores,
            k
        )
    })

comparison = pd.DataFrame(results)

comparison["Model_minus_Baseline"] = (
    comparison["LogisticRegression_Precision@K"]
    - comparison["Baseline_Precision@K"]
)

display(comparison)

,K,Baseline_Precision@K,LogisticRegression_Precision@K,Model_minus_Baseline
0,10,0.20,0.10,-0.10
1,20,0.30,0.15,-0.15
2,50,0.26,0.22,-0.04


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

I inspected the highest-scored evaluation rows and compared the predicted risk with the observed March outcome.

The Logistic Regression model did not outperform the Week-4 inverse-volume baseline on the same grouped evaluation split. Precision@10 was 0.10 for the model versus 0.20 for the baseline, Precision@20 was 0.15 versus 0.30, and Precision@50 was 0.22 versus 0.26. The model therefore did not add ranking value over the simple baseline in this evaluation.

The model's top 20 contained 3 true positives, giving Precision@20 of 0.15. Several false positives had relatively low February traffic, which is consistent with the model assigning higher risk to low-volume pages. However, one notable false positive had 39,733 February impressions and 183 clicks while receiving a model score of 1.00. This shows that the learned model can produce extreme risk scores for pages that did not go dark in March.

The largest absolute standardized coefficients were for February GSC clicks and February GSC impressions. Their negative coefficients mean that, within this fitted model and after standardization, higher values were associated with lower predicted probability of going dark. Organic sessions also had a negative coefficient, while scroll events had a positive coefficient. Average search position had the smallest absolute coefficient. These are model associations, not causal effects, and should not be interpreted as proof that changing one feature would cause the March outcome to change.

The error analysis suggests that the simple inverse-volume baseline is a useful benchmark for this task. The Logistic Regression model did not improve Precision@K on this evaluation split, so adding model complexity did not provide a demonstrated ranking benefit here. This result should be reported rather than selecting a more complex model only because it is expected to perform better.

The grouped split also provides a stricter evaluation because the same client was not placed in both training and test data. The test set contains 6 clients and 12,962 content rows, while the training set contains 24 clients and 16,391 rows. Therefore, these results measure performance on unseen clients under this particular split and should not be generalized to every future client or month.

In [97]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Inspect the model's most highly ranked evaluation rows.

error_analysis = test_rows[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions_feb",
        "gsc_clicks_feb",
        "avg_position_feb",
        "sessions_organic_feb",
        "scroll_events_feb",
        "went_dark"
    ]
].copy()

error_analysis["model_score"] = model_scores

error_analysis = error_analysis.sort_values(
    "model_score",
    ascending=False
)

display(error_analysis.head(20))

,client_hash_id,content_hash_id,gsc_impressions_feb,gsc_clicks_feb,avg_position_feb,sessions_organic_feb,scroll_events_feb,went_dark,model_score
6865,client_23a62021009f63c4,content_5460a0e854602a47,39733.0,183.0,24.532706,453.0,3037.0,0,1.000000
9993,client_9958f0a7ae1df715,content_01ab236521ddf2f9,516.0,10.0,31.155039,18.0,102.0,0,0.987304
9985,client_9958f0a7ae1df715,content_ffadc4cfd8604d34,580.0,5.0,13.715517,10.0,47.0,0,0.767480
11948,client_9958f0a7ae1df715,content_6e74280277dc6d99,130.0,3.0,19.023077,13.0,21.0,0,0.430294
9975,client_9958f0a7ae1df715,content_d4411158e5c57b17,288.0,6.0,21.847222,8.0,28.0,0,0.419339
11845,client_9958f0a7ae1df715,content_49d620e62faef307,172.0,3.0,14.534884,3.0,13.0,0,0.329864
9958,client_9958f0a7ae1df715,content_737bf0dacc016080,247.0,3.0,16.599190,4.0,11.0,0,0.286742
9954,client_9958f0a7ae1df715,content_2f061930226ff4c1,120.0,5.0,18.016667,6.0,16.0,1,0.277789
9976,client_9958f0a7ae1df715,content_c04fa223bde1fcce,172.0,3.0,19.040698,4.0,8.0,0,0.248726
9973,client_9958f0a7ae1df715,content_1984f849589da224,163.0,3.0,13.251534,5.0,7.0,0,0.224840


In [98]:
top20 = error_analysis.head(20)

print("Top-20 predicted positives:", top20["went_dark"].sum())
print("Top-20 precision:", top20["went_dark"].mean())

print("\nFalse positives in the model's top 20:")

display(
    top20[top20["went_dark"] == 0][
        [
            "gsc_impressions_feb",
            "gsc_clicks_feb",
            "avg_position_feb",
            "sessions_organic_feb",
            "scroll_events_feb",
            "model_score",
            "went_dark"
        ]
    ]
)

Top-20 predicted positives: 3
Top-20 precision: 0.15

False positives in the model's top 20:


,gsc_impressions_feb,gsc_clicks_feb,avg_position_feb,sessions_organic_feb,scroll_events_feb,model_score,went_dark
6865,39733.0,183.0,24.532706,453.0,3037.0,1.000000,0
9993,516.0,10.0,31.155039,18.0,102.0,0.987304,0
9985,580.0,5.0,13.715517,10.0,47.0,0.767480,0
11948,130.0,3.0,19.023077,13.0,21.0,0.430294,0
9975,288.0,6.0,21.847222,8.0,28.0,0.419339,0
11845,172.0,3.0,14.534884,3.0,13.0,0.329864,0
9958,247.0,3.0,16.599190,4.0,11.0,0.286742,0
9976,172.0,3.0,19.040698,4.0,8.0,0.248726,0
9973,163.0,3.0,13.251534,5.0,7.0,0.224840,0
9988,372.0,4.0,29.330645,6.0,10.0,0.223383,0


In [99]:
logistic_model = model.named_steps["logistic"]

coefficients = pd.DataFrame({
    "feature": features,
    "coefficient": logistic_model.coef_[0]
})

coefficients["abs_coefficient"] = coefficients["coefficient"].abs()

coefficients = coefficients.sort_values(
    "abs_coefficient",
    ascending=False
)

display(coefficients)

,feature,coefficient,abs_coefficient
1,gsc_clicks_feb,-8.922111,8.922111
0,gsc_impressions_feb,-2.822656,2.822656
3,sessions_organic_feb,-0.344708,0.344708
4,scroll_events_feb,0.271616,0.271616
2,avg_position_feb,0.028705,0.028705


In [100]:
# Check the final frame against the exact ML-04 universe.

print("Final modeling rows:", len(final_frame))
print("Universe rows:", len(universe_feb))

# Check for duplicate client/content pairs
duplicates = final_frame.duplicated(
    subset=["client_hash_id", "content_hash_id"]
).sum()

print("Duplicate client/content pairs:", duplicates)

# Check missing values
print("\nMissing values:")
print(final_frame.isna().sum())

Final modeling rows: 29353
Universe rows: 29700
Duplicate client/content pairs: 0

Missing values:
client_hash_id              0
content_hash_id             0
gsc_impressions_feb         0
gsc_clicks_feb              0
avg_position_feb            0
sessions_organic_feb    14156
scroll_events_feb       14156
measured_days_mar           0
clicks_mar                  0
went_dark                   0
dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.